In [ ]:
%pip install -qU langgraph langsmith ipywidgets

In [ ]:
%pip install -qU langchain langchain_community langchain_groq 

In [ ]:
import os

groq_api_key = os.getenv("GROQ_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
#print(groq_api_key)
#print(langsmith_api_key)

In [ ]:
from langchain_groq import ChatGroq

#
###### qwen/qwen3-32b
#
###### openai/gpt-oss-20b
###### openai/gpt-oss-120b
###### openai/gpt-oss-safeguard-20b
#
###### whisper-large-v3
###### whisper-large-v3-turbo
#
###### llama-3.1-8b-instant
###### llama-3.3-70b-versatile
###### meta-llama/llama-4-scout-17b-16e-instruct
###### meta-llama/llama-prompt-guard-2-22m
###### meta-llama/llama-prompt-guard-2-86m
#
###### groq/compound
###### groq/compound-mini

In [ ]:


llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b")
llm

##### Chatbot Using Langgraph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
  messages:Annotated[list,add_messages]

graph_builder=StateGraph(State)


In [ ]:
graph_builder

In [ ]:
def chatbot(state:State):
  return {"messages":llm.invoke(state['messages'])}

In [ ]:
graph_builder.add_node("chatbot",chatbot)

In [ ]:
graph_builder

In [ ]:
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

In [ ]:
graph=graph_builder.compile()

In [ ]:
from IPython.display import Image, display
try:
  display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
  pass

In [ ]:
while True:
  user_input= input("User: ") #"User: Brijesh" #
  if user_input.lower() in ["quit","q"]:
    print("Good Bye")
    break
  for event in graph.stream({'messages':("user",user_input)}):
    print(event.values())
    for value in event.values():
      print(value['messages'])
      print("Assistant:",value["messages"].content)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
output = widgets.Output()

text_input = widgets.Text(
    value='',  # Initial 
    description='Type here:',  # Label for the text input
    disabled=False  #  to be editable
)

# Function to handle changes in the text input
def on_text_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            # from 'Output widgets: leveraging Jupyter’s display system' https://ipywidgets.readthedocs.io/en/latest/examples/Output%20Widget.html
            output.clear_output() 
            print(f"You entered: {change['new']}")

# Observe changes in the text input widget
text_input.observe(on_text_change, names='value')

display(output, text_input)

In [ ]:
input_user = input("user:")
print(input_user)